In [ ]:
import pandas as pd
import numpy as np
import torch

In [ ]:
df0 = pd.read_csv("dataset0.tsv", sep="\t", header=None, names=["usersha1", "artmbid", "artname", "plays"])
df1 = pd.read_csv("dataset1.csv")
df2 = pd.read_csv("dataset2.csv")
df3 = pd.read_csv("dataset3.csv")

In [ ]:
print(torch.cuda.is_available())

print(df0.columns)
print(df1.columns)
print(df2.columns)
print(df3.columns)

In [ ]:
print(", ".join(f"{col}: {dtype}" for col, dtype in df0.dtypes.items()))
print(", ".join(f"{col}: {dtype}" for col, dtype in df1.dtypes.items()))
print(", ".join(f"{col}: {dtype}" for col, dtype in df2.dtypes.items()))
print(", ".join(f"{col}: {dtype}" for col, dtype in df3.dtypes.items()))

# Data Layer

In [ ]:
in_df2 = df0['artmbid'].isin(df2['mbid'])
df2_names = set(df2['artist_lastfm'].str.lower().dropna()) | set(df2['artist_mb'].str.lower().dropna())
df1_match = df1['artist'].str.lower().isin(df2_names)
df3_match = df3['artist'].str.lower().isin(df2_names)

## Preprocessing

In [ ]:
artists = (
    df2[~df2['ambiguous_artist']]
    [['mbid', 'artist_lastfm', 'country_lastfm', 'tags_lastfm', 'listeners_lastfm', 'scrobbles_lastfm']]
    .rename(columns={'mbid': 'artist_mbid', 'artist_lastfm': 'name',
                     'country_lastfm': 'country', 'tags_lastfm': 'tags_detailed',
                     'listeners_lastfm': 'listeners', 'scrobbles_lastfm': 'scrobbles'})
    .dropna(subset=['artist_mbid'])
    .drop_duplicates('artist_mbid')
    .reset_index(drop=True)
)
artists['_name_lower'] = artists['name'].str.lower()

audio_cols = ['danceability', 'energy', 'valence', 'tempo', 'duration_ms', 'year']
df1_agg = (df1.groupby(df1['artist'].str.lower())[audio_cols]
           .mean().reset_index().rename(columns={'artist': '_name_lower'}))
artists = artists.merge(df1_agg, on='_name_lower', how='left')

emotion_cols = ['number_of_emotion_tags', 'valence_tags', 'arousal_tags', 'dominance_tags']
df3_agg = (df3.groupby(df3['artist'].str.lower())[emotion_cols]
           .mean().reset_index().rename(columns={'artist': '_name_lower'}))
artists = artists.merge(df3_agg, on='_name_lower', how='left').drop(columns=['_name_lower'])

In [ ]:
df0_filtered = df0[df0['artmbid'].isin(artists['artist_mbid'])].copy()
interacted_mbids = df0_filtered['artmbid'].unique()
artists = artists[artists['artist_mbid'].isin(interacted_mbids)].reset_index(drop=True)
user_to_id   = {u: i for i, u in enumerate(df0_filtered['usersha1'].unique())}
artist_to_id = {a: i for i, a in enumerate(artists['artist_mbid'])}
df0_filtered['user_id']   = df0_filtered['usersha1'].map(user_to_id)
df0_filtered['artist_id'] = df0_filtered['artmbid'].map(artist_to_id)
artists['artist_id']      = artists['artist_mbid'].map(artist_to_id)
interactions = df0_filtered[['user_id', 'artist_id', 'plays']].reset_index(drop=True)
n_users   = len(user_to_id)
n_artists = len(artists)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(interactions, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

## Modeling

In [ ]:
import os, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.preprocessing import StandardScaler

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

FEAT_GROUPS = {
    'none':       [],
    'audio':      ['danceability', 'energy', 'valence', 'tempo', 'duration_ms', 'year'],
    'emotion':    ['valence_tags', 'arousal_tags', 'dominance_tags', 'number_of_emotion_tags'],
    'popularity': ['listeners', 'scrobbles'],
    'all':        ['danceability', 'energy', 'valence', 'tempo', 'duration_ms', 'year',
                   'valence_tags', 'arousal_tags', 'dominance_tags', 'number_of_emotion_tags',
                   'listeners', 'scrobbles'],
}

artist_lookup = artists.set_index('artist_id').sort_index()

def build_feat_tensor(cols):
    if not cols:
        return None
    mat = artist_lookup[cols].fillna(0).values.astype('float32')
    for i, c in enumerate(cols):
        if c in ('listeners', 'scrobbles', 'duration_ms'):
            mat[:, i] = np.log1p(mat[:, i])
    mat = StandardScaler().fit_transform(mat).astype('float32')
    return torch.FloatTensor(mat).to(device)

feat_tensors = {name: build_feat_tensor(cols) for name, cols in FEAT_GROUPS.items()}
for name, t in feat_tensors.items():
    print(f"  feat[{name!r:12}]: {t.shape if t is not None else None}")

### Evaluation

In [ ]:
train_pos_set = train_df.groupby('user_id')['artist_id'].agg(set).to_dict()
val_pos_set   = val_df.groupby('user_id')['artist_id'].agg(set).to_dict()

def sampled_eval(score_fn, test_df, ks=(10, 20), n_eval=5000, n_neg=99, seed=5235):
    rng      = np.random.default_rng(seed)
    by_user  = test_df.groupby('user_id')['artist_id'].agg(list).to_dict()
    eval_users = list(by_user.keys())
    if len(eval_users) > n_eval:
        eval_users = rng.choice(eval_users, n_eval, replace=False).tolist()

    out = {f'Recall@{k}': [] for k in ks}
    out.update({f'nDCG@{k}': [] for k in ks})

    for u in eval_users:
        pos  = by_user[u]
        excl = (train_pos_set.get(u, set()) | val_pos_set.get(u, set()) | set(pos))

        negs, seen = [], set()
        while len(negs) < n_neg:
            batch = rng.integers(0, n_artists, n_neg * 4).tolist()
            for c in batch:
                if c not in excl and c not in seen:
                    negs.append(c); seen.add(c)
                if len(negs) == n_neg:
                    break

        cands = pos + negs
        u_t   = torch.full((len(cands),), u, dtype=torch.long, device=device)
        a_t   = torch.tensor(cands,          dtype=torch.long, device=device)
        with torch.no_grad():
            scores = score_fn(u_t, a_t).cpu().numpy()

        order   = np.argsort(-scores)
        pos_set = set(range(len(pos)))
        for k in ks:
            topk = set(order[:k])
            hits = len(topk & pos_set)
            out[f'Recall@{k}'].append(hits / min(len(pos), k))
            dcg  = sum(1/np.log2(r+2) for r, i in enumerate(order[:k]) if i in pos_set)
            idcg = sum(1/np.log2(i+2) for i in range(min(len(pos), k)))
            out[f'nDCG@{k}'].append(dcg / idcg if idcg else 0)

    return {m: float(np.mean(v)) for m, v in out.items()}

### Factorization Machine

In [ ]:
class FM(nn.Module):
    def __init__(self, n_users, n_artists, k=64, n_side=0):
        super().__init__()
        self.user_emb    = nn.Embedding(n_users,   k)
        self.artist_emb  = nn.Embedding(n_artists, k)
        self.user_bias   = nn.Embedding(n_users,   1)
        self.artist_bias = nn.Embedding(n_artists, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))
        self.n_side = n_side
        if n_side > 0:
            self.side_proj = nn.Linear(n_side, k, bias=False)
        nn.init.normal_(self.user_emb.weight,   std=0.01)
        nn.init.normal_(self.artist_emb.weight, std=0.01)

    def forward(self, u_ids, a_ids, side=None):
        u = self.user_emb(u_ids)
        a = self.artist_emb(a_ids)
        parts = [u, a]
        if self.n_side > 0 and side is not None:
            parts.append(self.side_proj(side))
        stacked = torch.stack(parts, dim=1)
        inter   = 0.5 * (stacked.sum(1).pow(2) - stacked.pow(2).sum(1)).sum(1)
        bias    = (self.user_bias(u_ids).squeeze(1)
                   + self.artist_bias(a_ids).squeeze(1)
                   + self.global_bias)
        return inter + bias

def train_fm(feat_name='none', k=64, epochs=5, batch_size=8192, lr=1e-3,
             sample_size=500_000):
    feat   = feat_tensors[feat_name]
    n_side = feat.shape[1] if feat is not None else 0
    model  = FM(n_users, n_artists, k=k, n_side=n_side).to(device)
    opt    = torch.optim.Adam(model.parameters(), lr=lr)

    pos_u = torch.LongTensor(train_df['user_id'].values).to(device)
    pos_a = torch.LongTensor(train_df['artist_id'].values).to(device)
    N     = len(pos_u)

    for ep in range(1, epochs + 1):
        model.train()
        idx = torch.randperm(N, device=device)[:sample_size]
        epoch_loss = 0.0
        for s in range(0, len(idx), batch_size):
            chunk = idx[s:s + batch_size]
            u  = pos_u[chunk];  ap = pos_a[chunk]
            an = torch.randint(0, n_artists, (len(chunk),), device=device)
            sp = model(u, ap, feat[ap] if feat is not None else None)
            sn = model(u, an, feat[an] if feat is not None else None)
            loss = F.softplus(sn - sp).mean()
            opt.zero_grad(); loss.backward(); opt.step()
            epoch_loss += loss.item()
        n_batches = (len(idx) + batch_size - 1) // batch_size
        print(f"  ep {ep}/{epochs}  loss={epoch_loss / n_batches:.4f}")

    model.eval()
    def score_fn(u_ids, a_ids):
        side = feat[a_ids] if feat is not None else None
        return model(u_ids, a_ids, side)
    return score_fn

### GNN & Graph Transformer

In [ ]:
from torch_geometric.data import HeteroData

train_u_t = torch.LongTensor(train_df['user_id'].values)
train_a_t = torch.LongTensor(train_df['artist_id'].values)

base_data = HeteroData()
base_data['user'].num_nodes    = n_users
base_data['artist'].num_nodes  = n_artists
base_data['user',   'listens',     'artist'].edge_index = torch.stack([train_u_t, train_a_t])
base_data['artist', 'rev_listens', 'user'  ].edge_index = torch.stack([train_a_t, train_u_t])
base_data = base_data.to(device)
print(base_data)

In [ ]:
from torch_geometric.nn import HeteroConv, SAGEConv, HGTConv

class GNNRec(nn.Module):
    def __init__(self, n_users, n_artists, hidden=64, n_layers=2,
                 n_artist_feats=0, metadata=None):
        super().__init__()
        self.user_emb   = nn.Embedding(n_users,   hidden)
        self.artist_emb = nn.Embedding(n_artists, hidden)
        nn.init.normal_(self.user_emb.weight,   std=0.1)
        nn.init.normal_(self.artist_emb.weight, std=0.1)
        self.n_artist_feats = n_artist_feats
        if n_artist_feats > 0:
            self.artist_proj = nn.Linear(n_artist_feats, hidden, bias=False)
        self.convs = nn.ModuleList([
            HeteroConv({
                ('user',   'listens',     'artist'): SAGEConv((-1, -1), hidden),
                ('artist', 'rev_listens', 'user'):   SAGEConv((-1, -1), hidden),
            }, aggr='sum') for _ in range(n_layers)
        ])

    def encode(self, data, artist_feat=None):
        dev = self.user_emb.weight.device
        if hasattr(data['user'], 'n_id'):   # mini-batch
            n_id_u = data['user'].n_id.to(dev)
            n_id_a = data['artist'].n_id.to(dev)
            x_u = self.user_emb(n_id_u)
            x_a = self.artist_emb(n_id_a)
            if self.n_artist_feats > 0 and artist_feat is not None:
                x_a = x_a + self.artist_proj(artist_feat[n_id_a])
        else:                               # full-graph
            x_u = self.user_emb.weight
            x_a = self.artist_emb.weight
            if self.n_artist_feats > 0 and artist_feat is not None:
                x_a = x_a + self.artist_proj(artist_feat)
        x = {'user': x_u, 'artist': x_a}
        for conv in self.convs:
            x = conv(x, data.edge_index_dict)
            x = {k: F.relu(v) for k, v in x.items()}
        return x['user'], x['artist']


class HGTRec(nn.Module):
    def __init__(self, n_users, n_artists, hidden=64, n_layers=2,
                 n_artist_feats=0, metadata=None, n_heads=4):
        super().__init__()
        self.user_emb   = nn.Embedding(n_users,   hidden)
        self.artist_emb = nn.Embedding(n_artists, hidden)
        nn.init.normal_(self.user_emb.weight,   std=0.1)
        nn.init.normal_(self.artist_emb.weight, std=0.1)
        self.n_artist_feats = n_artist_feats
        if n_artist_feats > 0:
            self.artist_proj = nn.Linear(n_artist_feats, hidden, bias=False)
        meta = metadata or base_data.metadata()
        self.convs = nn.ModuleList([
            HGTConv(hidden, hidden, meta, heads=n_heads) for _ in range(n_layers)
        ])

    def encode(self, data, artist_feat=None):
        dev = self.user_emb.weight.device
        if hasattr(data['user'], 'n_id'):
            n_id_u = data['user'].n_id.to(dev)
            n_id_a = data['artist'].n_id.to(dev)
            x_u = self.user_emb(n_id_u)
            x_a = self.artist_emb(n_id_a)
            if self.n_artist_feats > 0 and artist_feat is not None:
                x_a = x_a + self.artist_proj(artist_feat[n_id_a])
        else:
            x_u = self.user_emb.weight
            x_a = self.artist_emb.weight
            if self.n_artist_feats > 0 and artist_feat is not None:
                x_a = x_a + self.artist_proj(artist_feat)
        x = {'user': x_u, 'artist': x_a}
        for conv in self.convs:
            x = conv(x, data.edge_index_dict)
            x = {k: F.relu(v) for k, v in x.items()}
        return x['user'], x['artist']

In [ ]:
def train_graph_model(model_cls, feat_name='none', hidden=32, epochs=10,
                      sample_size=200_000, lr=1e-3, **model_kwargs):
    feat   = feat_tensors[feat_name]
    n_side = feat.shape[1] if feat is not None else 0
    model  = model_cls(n_users, n_artists, hidden=hidden, n_artist_feats=n_side,
                       metadata=base_data.metadata(), **model_kwargs).to(device)
    opt    = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    pos_u = torch.LongTensor(train_df['user_id'].values).to(device)
    pos_a = torch.LongTensor(train_df['artist_id'].values).to(device)
    N     = len(pos_u)

    for ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()

        u_embs, a_embs = model.encode(base_data, feat)

        idx  = torch.randperm(N, device=device)[:sample_size]
        u_s  = pos_u[idx]
        ap_s = pos_a[idx]
        an_s = torch.randint(0, n_artists, (len(idx),), device=device)

        sp   = (u_embs[u_s] * a_embs[ap_s]).sum(1)
        sn   = (u_embs[u_s] * a_embs[an_s]).sum(1)
        loss = F.softplus(sn - sp).mean()
        loss_val = loss.item()

        loss.backward()
        opt.step()

        del u_embs, a_embs, sp, sn, loss, idx, u_s, ap_s, an_s
        torch.cuda.empty_cache()

        print(f"  ep {ep}/{epochs}  loss={loss_val:.4f}")

    model.eval()
    with torch.no_grad():
        u_embs, a_embs = model.encode(base_data, feat)
    u_embs = u_embs.detach()
    a_embs = a_embs.detach()

    def score_fn(u_ids, a_ids):
        return (u_embs[u_ids] * a_embs[a_ids]).sum(1)
    
    del model
    torch.cuda.empty_cache()
    
    return score_fn

### Experiments

In [ ]:
results = {}
KS      = (10, 20)
REGIMES = list(FEAT_GROUPS.keys())   # none, audio, emotion, popularity, all

In [ ]:
print("Factorization Machine")
for regime in REGIMES:
    print(f"{regime}")
    sfn = train_fm(feat_name=regime, k=64, epochs=100, sample_size=200_000)
    results[('FM', regime)] = sampled_eval(sfn, test_df, ks=KS)
    print(" ", results[('FM', regime)])

In [ ]:
print("Graph Neural Network")
for regime in REGIMES:
    print(f"{regime}")
    sfn = train_graph_model(GNNRec, feat_name=regime, hidden=16, epochs=100,
                            sample_size=200_000)
    results[('GNN', regime)] = sampled_eval(sfn, test_df, ks=KS)
    print(" ", results[('GNN', regime)])

In [ ]:
torch.cuda.empty_cache()

In [ ]:
print("Graph Transformer (HGT)")
for regime in REGIMES:
    print(f"{regime}")
    sfn = train_graph_model(HGTRec, feat_name=regime, hidden=16, epochs=100,
                            sample_size=200_000, n_heads=2, n_layers=1)
    results[('HGT', regime)] = sampled_eval(sfn, test_df, ks=KS)
    print(" ", results[('HGT', regime)])

In [ ]:
import matplotlib.pyplot as plt

rows = [{'Model': m, 'Regime': r, **v} for (m, r), v in results.items()]
results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False, float_format='{:.4f}'.format))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
models  = ['FM', 'GNN', 'HGT']
for ax, metric in zip(axes, ['Recall@20', 'nDCG@20']):
    x = np.arange(len(REGIMES))
    w = 0.25
    for i, mdl in enumerate(models):
        vals = [results.get((mdl, r), {}).get(metric, 0) for r in REGIMES]
        ax.bar(x + i * w, vals, w, label=mdl)
    ax.set_xticks(x + w); ax.set_xticklabels(REGIMES, rotation=15)
    ax.set_title(metric); ax.legend()
plt.tight_layout()
plt.savefig('results.png', dpi=150)
plt.show()

In [ ]:
import re

def parse_losses(text):
    losses = []
    
    for line in text.strip().split("\n"):
        match = re.search(r"loss=([0-9.]+)", line)
        if match:
            losses.append(float(match.group(1)))
    
    return losses


def read_losses_from_file(filepath):
    with open(filepath, "r") as f:
        text = f.read()
    
    return parse_losses(text)

In [ ]:
l_FM_none = read_losses_from_file("l_fm_none.txt")
l_FM_audio = read_losses_from_file("l_fm_audio.txt")
l_FM_emotion = read_losses_from_file("l_fm_emotion.txt")
l_FM_popularity = read_losses_from_file("l_fm_popularity.txt")
l_FM_all = read_losses_from_file("l_fm_all.txt")

l_GNN_none = read_losses_from_file("l_gnn_none.txt")
l_GNN_audio = read_losses_from_file("l_gnn_audio.txt")
l_GNN_emotion = read_losses_from_file("l_gnn_emotion.txt")
l_GNN_popularity = read_losses_from_file("l_gnn_popularity.txt")
l_GNN_all = read_losses_from_file("l_gnn_all.txt")

l_HGT_none = read_losses_from_file("l_hgt_none.txt")
l_HGT_audio = read_losses_from_file("l_hgt_audio.txt")
l_HGT_emotion = read_losses_from_file("l_hgt_emotion.txt")
l_HGT_popularity = read_losses_from_file("l_hgt_popularity.txt")
l_HGT_all = read_losses_from_file("l_hgt_all.txt")

In [ ]:
def plot_loss_regimes(model, l_none, l_audio, l_emotion, l_popularity, l_all):
    epochs = range(1, len(l_none) + 1)

    plt.figure(figsize=(10, 6))

    plt.plot(epochs, l_none, label=f"{model} None", linewidth=3)
    plt.plot(epochs, l_audio, label=f"{model} Audio", linewidth=3)
    plt.plot(epochs, l_emotion, label=f"{model} Emotion", linewidth=3)
    plt.plot(epochs, l_popularity, label=f"{model} Popularity", linewidth=3)
    plt.plot(epochs, l_all, label=f"{model} All", linewidth=3)

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model} -- Loss Over Epochs")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_loss_regimes("FM", l_FM_none, l_FM_audio, l_FM_emotion, l_FM_popularity, l_FM_all)
plot_loss_regimes("GNN", l_GNN_none, l_GNN_audio, l_GNN_emotion, l_GNN_popularity, l_GNN_all)
plot_loss_regimes("HGT", l_HGT_none, l_HGT_audio, l_HGT_emotion, l_HGT_popularity, l_HGT_all)

### Model Expressivity

Linear-probe auxiliary tasks assess whether artist embeddings encode meaningful semantic structure beyond ranking.  Three tasks:
- **Genre** – most common genre per artist (from df1 track genres; ~5% artist coverage)
- **Country** – artist's primary country (from df2; ~45% coverage)
- **Popularity tier** – 5-quantile bucket of log-listeners (from df2; high coverage)

Each model is trained under `none` (graph structure only) and `all` (all side-info) regimes. Artist embeddings are then probed with a linear classifier (logistic regression). Higher Macro-F1 → the embedding encodes more of that property.

In [ ]:
genre_by_artist = (
    df1[['artist', 'genre']].dropna(subset=['genre'])
    .assign(_name_lower=lambda d: d['artist'].str.lower())
    .groupby('_name_lower')['genre']
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
    .rename(columns={'genre': 'top_genre'})
)
artists_ext = artists.copy()
artists_ext['_name_lower'] = artists_ext['name'].str.lower()
artists_ext = artists_ext.merge(genre_by_artist, on='_name_lower', how='left')

top_genres   = artists_ext['top_genre'].value_counts().head(20).index.tolist()
genre_mask   = artists_ext['top_genre'].isin(top_genres)
genre_art_ids = artists_ext.loc[genre_mask, 'artist_id'].values
genre_labels  = artists_ext.loc[genre_mask, 'top_genre'].map(
                    {g: i for i, g in enumerate(top_genres)}).values.astype(int)

artists_ext['primary_country'] = (
    artists_ext['country'].str.split(';').str[0].str.strip()
)
top_countries   = artists_ext['primary_country'].value_counts().head(20).index.tolist()
country_mask    = artists_ext['primary_country'].isin(top_countries)
country_art_ids = artists_ext.loc[country_mask, 'artist_id'].values
country_labels  = artists_ext.loc[country_mask, 'primary_country'].map(
                      {c: i for i, c in enumerate(top_countries)}).values.astype(int)

pop_mask      = artists_ext['listeners'].notna() & (artists_ext['listeners'] > 0)
pop_art_ids   = artists_ext.loc[pop_mask, 'artist_id'].values
pop_labels    = pd.qcut(
                    np.log1p(artists_ext.loc[pop_mask, 'listeners'].values),
                    q=5, labels=False).astype(int)

EXPR_TASKS = {
    'genre':   (genre_art_ids,   genre_labels),
    'country': (country_art_ids, country_labels),
    'pop_tier':(pop_art_ids,     pop_labels),
}

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler as SkScaler
from sklearn.model_selection import train_test_split as sk_split
from sklearn.metrics import f1_score

def probe(embs_np, art_ids, labels, max_iter=500):
    X = embs_np[art_ids]
    y = labels
    counts = np.bincount(y)
    valid  = np.where(counts >= 2)[0]
    mask   = np.isin(y, valid)
    X, y   = X[mask], y[mask]

    remap  = {old: new for new, old in enumerate(np.unique(y))}
    y      = np.array([remap[v] for v in y])

    X_tr, X_te, y_tr, y_te = sk_split(X, y, test_size=0.2, random_state=42, stratify=y)
    sc = SkScaler(); X_tr = sc.fit_transform(X_tr); X_te = sc.transform(X_te)

    clf = LogisticRegression(max_iter=max_iter, C=1.0, solver='lbfgs',
                             n_jobs=-1)
    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_te)
    return {'accuracy': float(clf.score(X_te, y_te)),
            'macro_f1': float(f1_score(y_te, preds, average='macro'))}


def get_fm_embeddings(feat_name='none', k=64, epochs=50,
                      batch_size=8192, lr=1e-3, sample_size=200_000):
    feat   = feat_tensors[feat_name]
    n_side = feat.shape[1] if feat is not None else 0
    model  = FM(n_users, n_artists, k=k, n_side=n_side).to(device)
    opt    = torch.optim.Adam(model.parameters(), lr=lr)
    pos_u  = torch.LongTensor(train_df['user_id'].values).to(device)
    pos_a  = torch.LongTensor(train_df['artist_id'].values).to(device)
    N      = len(pos_u)
    for ep in range(1, epochs + 1):
        model.train()
        idx = torch.randperm(N, device=device)[:sample_size]
        for s in range(0, len(idx), batch_size):
            chunk = idx[s:s + batch_size]
            u  = pos_u[chunk]; ap = pos_a[chunk]
            an = torch.randint(0, n_artists, (len(chunk),), device=device)
            sp = model(u, ap, feat[ap] if feat is not None else None)
            sn = model(u, an, feat[an] if feat is not None else None)
            loss = F.softplus(sn - sp).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        if ep % 10 == 0:
            print(f"  ep {ep}/{epochs}")
    model.eval()
    embs = model.artist_emb.weight.detach().cpu().numpy()
    del model; torch.cuda.empty_cache()
    return embs

def get_graph_embeddings(model_cls, feat_name='none', hidden=32, epochs=50,
                         sample_size=200_000, lr=1e-3, **model_kwargs):
    feat   = feat_tensors[feat_name]
    n_side = feat.shape[1] if feat is not None else 0
    model  = model_cls(n_users, n_artists, hidden=hidden, n_artist_feats=n_side,
                       metadata=base_data.metadata(), **model_kwargs).to(device)
    opt    = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    pos_u  = torch.LongTensor(train_df['user_id'].values).to(device)
    pos_a  = torch.LongTensor(train_df['artist_id'].values).to(device)
    N      = len(pos_u)
    for ep in range(1, epochs + 1):
        model.train(); opt.zero_grad()
        u_embs, a_embs = model.encode(base_data, feat)
        idx  = torch.randperm(N, device=device)[:sample_size]
        u_s  = pos_u[idx]; ap_s = pos_a[idx]
        an_s = torch.randint(0, n_artists, (len(idx),), device=device)
        sp   = (u_embs[u_s] * a_embs[ap_s]).sum(1)
        sn   = (u_embs[u_s] * a_embs[an_s]).sum(1)
        loss = F.softplus(sn - sp).mean()
        loss.backward(); opt.step()
        del u_embs, a_embs, sp, sn, loss, idx, u_s, ap_s, an_s
        torch.cuda.empty_cache()
        if ep % 10 == 0:
            print(f"  ep {ep}/{epochs}")
    model.eval()
    with torch.no_grad():
        _, a_embs = model.encode(base_data, feat)
    embs = a_embs.detach().cpu().numpy()
    del model, a_embs; torch.cuda.empty_cache()
    return embs

In [ ]:
EXPR_REGIMES = ['none', 'all']
expr_results  = {}

for feat_name in EXPR_REGIMES:
    embs = get_fm_embeddings(feat_name=feat_name, k=64, epochs=50, sample_size=200_000)
    for task, (a_ids, lbls) in EXPR_TASKS.items():
        r = probe(embs, a_ids, lbls)
        expr_results[('FM', feat_name, task)] = r

for feat_name in EXPR_REGIMES:
    embs = get_graph_embeddings(GNNRec, feat_name=feat_name, hidden=16, epochs=50,
                                sample_size=200_000)
    for task, (a_ids, lbls) in EXPR_TASKS.items():
        r = probe(embs, a_ids, lbls)
        expr_results[('GNN', feat_name, task)] = r

torch.cuda.empty_cache()

for feat_name in EXPR_REGIMES:
    embs = get_graph_embeddings(HGTRec, feat_name=feat_name, hidden=16, epochs=50,
                                sample_size=200_000, n_heads=2, n_layers=1)
    for task, (a_ids, lbls) in EXPR_TASKS.items():
        r = probe(embs, a_ids, lbls)
        expr_results[('HGT', feat_name, task)] = r

In [ ]:
expr_rows = [{'Model': m, 'Regime': reg, 'Task': t, **v}
             for (m, reg, t), v in expr_results.items()]
expr_df = pd.DataFrame(expr_rows)
print(expr_df.to_string(index=False, float_format='{:.4f}'.format))

tasks   = list(EXPR_TASKS.keys())
models  = ['FM', 'GNN', 'HGT']
colors  = ['steelblue', 'darkorange', 'seagreen']

fig, axes = plt.subplots(len(tasks), len(EXPR_REGIMES),
                         figsize=(10, 3.5 * len(tasks)), sharey='row')
for i, task in enumerate(tasks):
    for j, regime in enumerate(EXPR_REGIMES):
        ax = axes[i, j]
        vals = [expr_results.get((m, regime, task), {}).get('macro_f1', 0) for m in models]
        bars = ax.bar(models, vals, color=colors, edgecolor='white', linewidth=0.5)
        ax.set_title(f'{task}  [{regime}]', fontsize=10)
        ax.set_ylabel('Macro F1' if j == 0 else '')
        ax.set_ylim(0, max(vals) * 1.35 + 0.03)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=8)

fig.suptitle('Model Expressivity — Linear Probe Macro-F1', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('expressivity_results.png', dpi=150, bbox_inches='tight')
plt.show()